# 확장 시각화 — 텍스트 워드클라우드(WordCloud) · 지도 시각화(Folium)


**설명 전용 자료:** 개념·예제·주의점을 Markdown 셀로 제공합니다. 연습문제와 빈 풀이 셀은 같은 이름의 `_Practice.ipynb`에 있습니다.

**학습 목표:** 비정형 텍스트에서 단어 빈도를 추출해 워드클라우드(`WordCloud`)로 시각화하고, 위도·경도 좌표 데이터를 `folium.Map`, `Marker`, `MarkerCluster`로 지도에 표시하고 HTML 파일로 저장합니다.

## 처음 배우는 분을 위한 안내

토큰은 문장을 쪼갠 단어 단위, 불용어는 분석에서 빼기로 정한 단어, Counter는 출현 횟수를 세는 도구입니다. 여기서는 공백 분리만 사용하며 형태소 분석은 필요하지 않습니다. 위도는 남북 위치, 경도는 동서 위치이며 지도 좌표는 [위도, 경도] 순서입니다.

**학습 순서:** 문제 1~4는 필수 기초, 5~12는 선택 도전입니다. 5~6번은 한글 글꼴이, 7번 이후 지도 화면은 인터넷이 필요할 수 있습니다. 인터넷이 없어도 마커 데이터와 HTML 생성 여부는 확인할 수 있습니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

그래프 창이 비어 있으면 데이터 개수와 결측 제거 결과를 먼저 확인하세요. 한글이 네모로 나오면 글꼴 설정 문제이며 계산 오류와 구분합니다.

**지도 화면 주의:** 배경 지도와 Leaflet 스크립트는 외부 서버를 사용합니다. 네트워크가 막혀 빈 배경이 나와도 마커 개수·좌표와 HTML 저장 성공을 따로 확인하세요. 글꼴 파일이 없으면 5~6번 워드클라우드는 잠시 건너뛰고 1~4번 표와 7번 이후 지도를 진행할 수 있습니다.

## 실행 환경과 예제 데이터

워드클라우드와 지도 시각화에는 `wordcloud`와 `folium` 라이브러리가 필요합니다. 없는 경우 `%pip install wordcloud folium`을 먼저 실행하세요. 한글 워드클라우드는 `font_path`에 한글 글꼴 파일 경로를 지정해야 글자가 깨지지 않습니다.

```python
import platform
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import folium
from folium.plugins import MarkerCluster

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
plt.rcParams['axes.unicode_minus'] = False

# 지정 글꼴이 없는 환경에서는 설치된 한글 글꼴 파일을 FONT_PATH에 지정합니다.
from pathlib import Path
if not Path(FONT_PATH).is_file():
    candidates = [p for pattern in ('/System/Library/Fonts/*Apple*','/usr/share/fonts/truetype/nanum/*.ttf') for p in __import__('glob').glob(pattern) if Path(p).is_file()]
    if candidates:
        FONT_PATH = candidates[0]
    else:
        raise FileNotFoundError('한글 글꼴 파일이 필요합니다. 글꼴 파일을 올린 뒤 FONT_PATH를 해당 경로로 설정하세요.')

# 1. 워드클라우드 실습용 고객 리뷰·문장 리스트
review_lines = [
    "배송 빠름 포장 깔끔 가격 만족 합니다.",
    "배송 빠름 품질 최고 재구매 의사 있습니다,",
    "가격 합리적 배송 빠름 상담 친절 합니다.",
    "포장 깔끔 품질 만족 배송 최고 입니다.",
    "품질 최고 가격 만족 포장 안전 합니다."
]

# 2. Folium 지도 실습용 매장 위치 데이터 (국가 및 위경도 포함, 8행)
stores_df = pd.DataFrame({
    'Store.Name': [
        '시청역점', '광화문점', '을지로입구점', '명동역점',
        '정동점', '강남역점', '부산서면점', '도쿄신주쿠점'
    ],
    'Country': ['KR', 'KR', 'KR', 'KR', 'KR', 'KR', 'KR', 'JP'],
    'Latitude': [37.5658, 37.5715, 37.5660, 37.5609, 37.5672, 37.4979, 35.1578, 35.6938],
    'Longitude': [126.9772, 126.9765, 126.9827, 126.9858, 126.9721, 127.0276, 129.0592, 139.7034]
})

# 3. 공공시설(의류수거함) 위치 현황 가상 데이터 (결측 1행 포함, 6행)
collection_bins = pd.DataFrame({
    'bin_id': ['B01', 'B02', 'B03', 'B04', 'B05', 'B06'],
    'dong': ['둔산동', '둔산동', '갈마동', '월평동', '갈마동', '둔산동'],
    'latitude': [36.3504, 36.3521, 36.3482, 36.3585, None, 36.3512],
    'longitude': [127.3845, 127.3880, 127.3720, 127.3615, None, 127.3860]
})
```

## 텍스트 정제와 `Counter` 단어 빈도 계산

문장 데이터에서 워드클라우드를 만들려면 먼저 문장을 단어로 쪼개고(토큰화), 마침표·쉼표 같은 문장부호와 불필요한 어미(`합니다`, `입니다` 등)를 정리한 뒤 단어별 등장 횟수를 집계해야 합니다.

| 단계 | 주요 도구 / 메서드 | 역할 |
| --- | --- | --- |
| 1. 단어 분리 | `row.split()` | 공백(띄어쓰기·탭) 기준으로 문장을 단어 리스트로 분할 |
| 2. 문장부호·어미 정리 | `word.strip('.,!?"\'')` | 단어 양끝의 마침표, 쉼표 등 불필요 기호 제거 |
| 3. 빈도 집계 | `collections.Counter(words)` | `{단어: 출현빈도}` 형태의 딕셔너리 하위 객체 생성 |
| 4. 빈도순 정렬 | `sorted(counts.items(), key=lambda x: x[1], reverse=True)` | 빈도(`x[1]`) 기준 내림차순으로 `(단어, 빈도)` 튜플 정렬 |
| 5. DataFrame → dict | `df.set_index('word')['freq'].to_dict()` | 표 형태로 점검한 빈도 데이터를 `WordCloud` 입력용 딕셔너리로 변환 |

```python
tokens = []
stopwords = {'합니다', '입니다', '있습니다'}
for line in review_lines:
    for raw in line.split():
        w = raw.strip('.,!')
        if w and w not in stopwords:
            tokens.append(w)

word_counts = Counter(tokens)
wc_sorted = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
wcdf = pd.DataFrame(wc_sorted, columns=['word', 'freq'])
word_freq = wcdf.set_index('word')['freq'].to_dict()
print(word_freq)
```

## 워드클라우드(`WordCloud`) 생성과 출력

`WordCloud().generate_from_frequencies(word_freq)`는 `{단어: 빈도}` 딕셔너리를 받아 빈도가 클수록 글자 크기를 크게 배치한 이미지를 생성합니다.

- **주요 옵션:** `font_path`(한글 폰트 파일 경로), `width`·`height`(픽셀 해상도), `background_color`(배경색, 기본 `'black'`, 보고서용은 `'white'` 권장), `colormap`(`'viridis'`, `'autumn'`, `'coolwarm'` 등 색상 테마).
- 생성된 객체는 `plt.imshow(wordcloud)`로 화면에 그리고, 픽셀 좌표 축이 보이지 않도록 `plt.axis('off')`를 함께 호출합니다.

```python
wc = WordCloud(
    font_path=FONT_PATH,
    width=800,
    height=500,
    background_color='white',
    colormap='autumn'
).generate_from_frequencies(word_freq)

plt.figure(figsize=(7, 4))
plt.imshow(wc)
plt.axis('off')
plt.title('고객 리뷰 핵심 키워드 워드클라우드')
plt.show()
```

## Folium 지도 생성(`folium.Map`)과 마커(`folium.Marker`)

Folium은 위도(`Latitude`, 북위 ~37°)와 경도(`Longitude`, 동경 ~127°) 좌표를 받아 인터랙티브 웹 지도(Leaflet.js 기반)를 생성합니다.

| 구성 요소 | 주요 파라미터 | 설명 |
| --- | --- | --- |
| `folium.Map` | `location=[위도, 경도]`, `zoom_start=14`, `tiles=...` | 중심 좌표와 초기 확대 배율, 배경 지도 스타일 지정 |
| `tiles` 옵션 | `'OpenStreetMap'`(기본), `'CartoDB positron'`, `'Esri.WorldImagery'`(위성) | 목적에 맞는 배경 타일 선택 (외부 타일 서버 상황에 따라 변경 가능) |
| `folium.Marker` | `location=[위도, 경도]`, `tooltip=...`, `popup=...`, `icon=folium.Icon(color='red')` | 마우스 오버 시 `tooltip`, 클릭 시 `popup` 창을 띄우는 핀 마커 추가 |

```python
city_hall = [37.5662952, 126.9779451]
m = folium.Map(location=city_hall, zoom_start=14, tiles='OpenStreetMap')

folium.Marker(
    location=city_hall,
    tooltip='서울시청',
    popup=folium.Popup('서울시청 위치', max_width=200),
    icon=folium.Icon(color='red')
).add_to(m)
```

## 위경도 범위 필터링, `MarkerCluster`, HTML 저장

수백~수천 개의 매장 데이터를 지도에 그대로 찍으면 마커가 겹쳐 지도가 보이지 않고 브라우저가 느려집니다.

1. **관심 영역 필터링(`between`):** 국가 코드(`Country == 'KR'`)와 위도·경도 사각형 구간(`Latitude.between(37.55, 37.58) & Longitude.between(126.96, 126.99)`)으로 대상 지역의 데이터만 먼저 추립니다.
2. **`MarkerCluster` 군집화:** 가까운 마커들을 원형 숫자 배지로 묶어 보여주고, 지도를 확대하면 개별 마커로 펼쳐지도록 `cluster = MarkerCluster().add_to(m)`에 마커를 추가합니다.
3. **`iterrows()` 반복 추가와 `m.save('map.html')`:** 필터링된 데이터프레임을 `for _, row in stores.iterrows():`로 순회하며 마커를 등록하고, 완성된 지도는 `.save('파일명.html')`로 저장해 웹 브라우저에서 공유·열람합니다.

```python
seoul_stores = stores_df[
    (stores_df['Country'] == 'KR')
    & stores_df['Latitude'].between(37.55, 37.58)
    & stores_df['Longitude'].between(126.96, 126.99)
][['Store.Name', 'Latitude', 'Longitude']].copy()

cluster = MarkerCluster(name='Stores').add_to(m)
for _, row in seoul_stores.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        tooltip=row['Store.Name'],
        popup=folium.Popup(row['Store.Name'], max_width=250)
    ).add_to(cluster)

m.save('seoul_stores_map.html')
```

## 선택 도전: 단어 네 개를 표로 검산하기

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
extra_words = ['배송','배송','가격','포장']
```

### 진행 순서

1. Counter로 단어 출현 횟수를 셉니다.
2. 빈도 합계와 고유 단어 수를 출력합니다.
3. 가장 자주 등장하는 단어를 적습니다. 이 선택 도전은 폰트·지도·외부 파일 없이 표만으로 완료합니다.

**완료 기준:** 배송 2회, 가격·포장 각각 1회. 빈도 합계 4, 고유 단어 3개.

### 선택 도전 풀이

```python
from collections import Counter
extra_counts=Counter(extra_words);print(extra_counts)
print(sum(extra_counts.values()),len(extra_counts))
assert extra_counts['배송']==2 and sum(extra_counts.values())==4 and len(extra_counts)==3
```

## 복습 순서

1. 문장 리스트에서 공백 분리(`split`) → 기호 제거(`strip`) → 불용어 제외 순으로 토큰 리스트를 만듭니다.
2. `Counter`와 `sorted`로 상위 빈도 단어를 확인한 뒤 `{단어: 빈도}` 딕셔너리로 변환해 `WordCloud`에 전달합니다.
3. `folium.Map`의 위도·경도 순서(`[위도, 경도]`)를 헷갈리지 않고 중심 좌표를 설정합니다.
4. `between()`으로 관심 좌표 범위를 필터링하고 `MarkerCluster`와 `iterrows()`로 다중 마커를 추가한 뒤 `.save()`로 HTML 파일을 생성합니다.